# Correção — data_entrega_real sempre igual à prevista (TMS)

Corrige um problema encontrado ao calcular gold_otif: `data_entrega_real` nascia sempre idêntica a `data_entrega_prevista` (sem variação real), tornando o cálculo de OTIF sempre 100% — sem chance estatística de dar outro resultado. `status_entrega` também era sorteado de forma independente da data, sem relação real entre os dois campos.

Correção aplicada em `simulador_tms.py`: `data_entrega_real` agora varia (80% no prazo, 15% atrasada 1-3 dias, 5% devolvida), e `status_entrega` é derivado dessa mesma decisão.

Este notebook regenera os dados de `tms_comprovantes_entrega` nas 4 datas já testadas, refletindo a correção.

In [0]:
%pip install dbldatagen Faker

In [0]:
dbutils.library.restartPython()

In [0]:
from src.ingestao.ingestor_autoloader import IngestorAutoloader

ingestor = IngestorAutoloader(spark=spark, sistema="tms", tabela="tms_comprovantes_entrega")
resultado = ingestor.executar()
print(resultado)

In [0]:
from src.transformacao.configuracao_tabelas import CONFIGURACAO_TABELAS
from src.transformacao.transformar_bronze_para_silver import transformar_bronze_para_silver

resultado = transformar_bronze_para_silver(
    spark=spark,
    catalog="poc_pulse_observability",
    tabela="tms_comprovantes_entrega",
    config=CONFIGURACAO_TABELAS["tms_comprovantes_entrega"],
)
print(resultado)

In [0]:
df_comp = spark.table("poc_pulse_observability.silver.tms_comprovantes_entrega")
df_rem = spark.table("poc_pulse_observability.silver.tms_remessas").select("remessa_id", "data_entrega_prevista")

df_check = df_comp.join(df_rem, "remessa_id")
df_check.groupBy("status_entrega").count().show()

total = df_check.count()
diferentes = df_check.filter(df_check.data_entrega_real != df_check.data_entrega_prevista).count()
print(f"Total: {total} | Com data diferente da prevista: {diferentes} ({diferentes/total:.1%})")

In [0]:
spark.sql("DROP TABLE IF EXISTS poc_pulse_observability.bronze.tms_remessas")
dbutils.fs.rm("/Volumes/poc_pulse_observability/landing/raw/_autoloader_checkpoint/tms_remessas", recurse=True)
dbutils.fs.rm("/Volumes/poc_pulse_observability/landing/raw/_autoloader_schema/tms_remessas", recurse=True)

print("Reset de tms_remessas concluído.")

In [0]:
ingestor = IngestorAutoloader(spark=spark, sistema="tms", tabela="tms_remessas")
resultado = ingestor.executar()
print(resultado)

In [0]:
resultado = transformar_bronze_para_silver(
    spark=spark,
    catalog="poc_pulse_observability",
    tabela="tms_remessas",
    config=CONFIGURACAO_TABELAS["tms_remessas"],
)
print(resultado)

In [0]:
df_comp = spark.table("poc_pulse_observability.silver.tms_comprovantes_entrega")
df_rem = spark.table("poc_pulse_observability.silver.tms_remessas").select("remessa_id", "data_entrega_prevista")

df_check = df_comp.join(df_rem, "remessa_id")
df_check.groupBy("status_entrega").count().show()

total = df_check.count()
diferentes = df_check.filter(df_check.data_entrega_real != df_check.data_entrega_prevista).count()
print(f"Total: {total} | Com data diferente da prevista: {diferentes} ({diferentes/total:.1%})")